# Embedding
![rag_embedding](figures/rag_embedding.png)

- 분할된 텍스트를 벡터 표현(임베딩 벡터)으로 변환한다.
- LangChain은 OpenAI, HuggingFace 등 다양한 임베딩 모델을 지원하며, 동일한 인터페이스로 사용할 수 있다.
- [임베딩모델의 메서드](https://reference.langchain.com/python/langchain/embeddings/#langchain.embeddings.init_embeddings)

    - **`embed_documents(texts: List[str])`**
        - 여러 문서를 받아 벡터화(임베딩)한다.
        - Context를 벡터화 할 때 사용한다.
    - **`embed_query(text: str)`**
        - 하나의 문자열(문서)을 받아 벡터화한다.
        - Query를 벡터화 할 때 사용한다.


In [62]:
docs = [
        "나는 고양이와 개 중 반려동물로 개를 키우고 싶습니다.",
        "이 강아지 품종은 진도개 입니다. 국제 표준으로 중대형견으로 분류되며 다리가 길어 체고가 높은 편에 속합니다.",
        "日本の市内バスの運賃は主に距離によって決まり、地域やバス会社によって異なる場合があります",                 # 일본의 시내버스 요금은 주로 거리에 따라 결정되며, 지역 및 버스 회사에 따라 다를 수 있습니다.
        "Bus fares in the United States vary from city to city, but are generally around $2.90 for a regular bus.", # 미국의 버스 요금은 도시마다 다르지만, 일반적으로 정기 버스의 경우 2.90달러 정도입니다.
        "광역버스 요금은 일반 3000원, 청소년은 1800원, 어린이 1500원 입니다.", 
]

In [63]:
from dotenv import load_dotenv

load_dotenv()

True

In [64]:
from langchain_openai import OpenAIEmbeddings
model_name = "text-embedding-3-small"
model_name = "text-embedding-3-large"
e_model = OpenAIEmbeddings(model=model_name)

In [65]:
# HuggingFace의 OpenSource 임베일 모델 사용.
from langchain_huggingface import HuggingFaceEmbeddings
model_id = "codefuse-ai/F2LLM-v2-1.7B"
e_model = HuggingFaceEmbeddings(model= model_id)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [66]:
# 문서들 임베딩 처리(Embedding Vector로 변환)
embedded_docs = e_model.embed_documents(docs)

In [67]:
print(type(embedded_docs), len(embedded_docs))  # 변환된 문서 타입, 개수
print(type(embedded_docs[0]), len(embedded_docs[0]))

# text-embedding-3-small 모델 -> 1536차원 embedding vector로 변환.
# text-embedding-3-large 모델 -> 3072차원 embedding vector로 변환.

<class 'list'> 5
<class 'list'> 2048


In [68]:
embedded_docs[0][:10]

[0.003387451171875,
 -0.039306640625,
 -0.000278472900390625,
 0.0169677734375,
 0.03662109375,
 -0.0732421875,
 0.0029296875,
 0.03564453125,
 0.00982666015625,
 0.0654296875]

In [69]:
# 질문 문자열
# query = "개와 고양이 중 무엇을 더 좋아하나요?"
query = "요새 시내버스 요금 얼마에요?"


In [70]:
# 질문은 embedding vector로 변환
query_vector = e_model.embed_query(query)
print(type(query_vector))
query_vector[:10]

<class 'list'>


[-0.0771484375,
 -0.123046875,
 -0.056884765625,
 -0.0047607421875,
 0.046630859375,
 0.013427734375,
 -0.0035858154296875,
 -0.059326171875,
 -0.00494384765625,
 -0.00262451171875]

In [71]:
#############################
# 질문과 유사한 문서들을 검색
# 유사도 계산
# - 거리기반: 유클리디안 거리, 멘하탄 거리
# - 방향기반: 코사인 유사도. -1(cos180) ~ 0(cos90) ~ 1(cos0)

import numpy as np

# 코사인유사도 = 두 벡터간의 내적 / 각 벡터의 크기의 곱
def cosin_similarity(vector1, vector2):
    """vector1과 vector2의 코사인 유사도 계산"""
    v1 = np.array(vector1)
    v2 = np.array(vector2)
    return (v1 @ v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [72]:
cosin_similarity(embedded_docs[1], query_vector)

np.float64(0.042626307219600944)

In [57]:
search_list = []
for i, doc_vector in enumerate(embedded_docs):
    print(f"{i+1}, {cosin_similarity(doc_vector, query_vector)}")
    search_list.append((i,cosin_similarity(doc_vector, query_vector)))

1, 0.050705752910166414
2, 0.042626307219600944
3, 0.5084174213884401
4, 0.4959444267279385
5, 0.5827645207035504


In [61]:
top_k = 3
doc_idx = [idx for idx, score in search_list[:3]]
doc_idx # 질문과 유사도가 높은 문서 top_k 개의 index

[4, 2, 3]

In [59]:
# 유사도가 높은 순서대로 정렬 (1에 가까울 수록 유사도가 크다)
search_list.sort(key=lambda x: x[1], reverse=True)

In [60]:
search_list

[(4, np.float64(0.5827645207035504)),
 (2, np.float64(0.5084174213884401)),
 (3, np.float64(0.4959444267279385)),
 (0, np.float64(0.050705752910166414)),
 (1, np.float64(0.042626307219600944))]

# 벡터 데이터베이스(Vector Database)

![rag_vector_store](figures/rag_vector_store.png)

- **벡터 데이터베이스는** 데이터를 고차원 벡터(임베딩)로 변환하여 저장하고, 벡터 간의 유사도를 기반으로 검색과 관리를 수행하는 특수한 형태의 데이터베이스이다.

- **주요 특징**
  - 텍스트, 이미지, 오디오 등의 비정형 데이터를 수치 벡터로 변환하여 저장
   - 코사인 유사도, 유클리드 거리 등을 이용한 벡터 간 유사도 계산을 통한 검색
   - 근사 최근접 이웃(Approximate Nearest Neighbor, ANN) 알고리즘을 통한 빠른 검색을 지원.

## 벡터 데이터베이스와 딥러닝
- 벡터 데이터베이스는 딥러닝 기술의 발전과 깊은 관련이 있다.
- 딥러닝 모델은 학습 과정에서 데이터의 특징을 추출하는 방법을 함께 학습한다. 충분한 데이터를 학습한 딥러닝 모델은 **데이터의 특성을 설명하는 특성 벡터(feature vector)를 효과적으로 생성**할 수 있다.
- 이때 추출된 특성 벡터는 고차원 데이터(RAW Data)를 저차원 공간에서 표현한 **임베딩 벡터**다.
    - > **임베딩**은 고차원 데이터를 저차원 공간으로 변환하여 표현하는 방법으로, 정보 손실을 최소화하면서 데이터 간의 의미 있는 관계를 벡터 공간에서 유지한다.
- 딥러닝 모델로 추출한 데이터의 특징(feature vector)을 임베딩 공간에 배치하면, 비슷한 데이터는 가까이, 그렇지 않은 데이터는 멀리 배치된다.
- 이러한 특성을 활용하면 임베딩 벡터 간의 거리를 계산해 유사한 데이터를 효과적으로 검색할 수 있다. 벡터 데이터베이스는 이러한 임베딩 벡터의 특성을 기반으로 개발되었다.
- 딥러닝 기술의 발전과 폭넓은 활용으로 임베딩 데이터의 사용이 증가하면서, 이를 저장하고 관리하는 기능에 특화된 데이터베이스에 대한 수요도 증가해 다양한 벡터 데이터베이스가 등장했다.

## LLM과 벡터 데이터베이스
- ChatGPT(LLM)의 등장 이후 벡터 데이터베이스는 폭발적인 주목을 받았다.
- 임베딩 벡터의 유사도를 기반으로 문서를 검색하는 RAG(Relevant Augmented Generation) 기술은 LLM의 환각(할루시네이션) 현상을 줄이고, LLM을 추가 학습하지 않고도 최신 정보를 효율적으로 활용할 수 있는 핵심 기법으로 자리 잡았다.
   


## 벡터 데이터베이스 종류
![img](figures/vector_database.png)

<<https://blog.det.life/why-you-shouldnt-invest-in-vector-databases-c0cd3f59d23c>>

### 주요 벡터 데이터베이스 종류
- **Qdrant**
    - Rust로 개발된 고성능 벡터 검색 엔진으로, 실시간 근사 최근접 이웃 검색을 제공한다.  
    - 벡터와 함께 메타데이터(payload) 필터링을 결합할 수 있어, 의미 검색과 조건 검색을 동시에 수행할 수 있다.
    - RAG·추천·시맨틱 검색 등 LLM 연계 시스템에 적합하다.
- **Pinecone**
    - 클라우드 기반의 완전 관리형 벡터 데이터베이스 서비스로, 간단한 API를 통해 벡터 데이터를 관리할 수 있다.  
    - 자동 확장성과 고가용성을 제공하며, 실시간 데이터 수집과 유사성 검색에 최적화되어 있다.
    - 가장 쉽게 시작할 수 있는 관리형 서비스를 제공한다.
- **Chroma**
    - 벡터 임베딩을 효율적으로 저장하고 검색할 수 있는 오픈소스 데이터베이스로, AI 및 머신러닝 애플리케이션에 최적화되어 있다.
    - 대규모 임베딩 저장에 최적화되어 있다.
- **FAISS**
    - Facebook AI에서 개발한 고성능 벡터 검색 라이브러리로, 고차원 벡터의 효율적인 유사성 검색을 위해 최적화되어 있다.
    - GPU를 활용해 계산 성능을 높이며, 벡터 양자화 기술을 활용하여 메모리 사용을 최적화한다.
    - 근사 최근접 이웃 검색(ANNS)에 최적화되어 있다.
- **Milvus**
    - 오픈소스 벡터 데이터베이스로, 대규모 벡터 데이터를 효율적으로 저장하고 검색할 수 있다.  
    - 분산 아키텍처를 채택하여 확장성이 뛰어나며, IVF_PQ, DiskANN 등 다양한 인덱싱 알고리즘을 지원한다.
    - 대규모 데이터셋 처리에 가장 적합한 솔루션이다.
- **Weaviate**
    - 오픈소스 벡터 데이터베이스로, 텍스트, 이미지, 오디오 등 다양한 비정형 데이터를 벡터로 저장하고 검색할 수 있다.  
    - GraphQL API를 통해 접근 가능하며, 내장된 머신러닝 모듈을 통해 가장 강력한 의미론적 검색 기능을 제공한다.
- **Elasticsearch**
    - HNSW 알고리즘을 사용하여 벡터 검색을 구현하는 검색 엔진이다.
    - 전통적인 검색 기능과 벡터 검색을 효과적으로 결합할 수 있어, 하이브리드 검색에 가장 적합하다.
- **PGVector**
    - PostgreSQL의 확장 모듈로, 벡터 데이터를 저장하고 유사성 검색을 수행할 수 있게 해준다.  
    - SQL과 통합된 벡터 연산이 가능하며, L2 거리, 코사인 거리, 내적 등 다양한 거리 측정 방식을 지원한다.


# Langchain - Vector Store 연동 
- Langchain은 다양한 벡터 데이터베이스와 연동할 수 있다.
- 벡터 데이터베이스 마다 API가 다르기 때문에, Langchain을 사용하면 동일한 interface로 사용할 수 있다.

## **VectorStore**
- Langchain이 지원하는 모든 벡터 데이터베이스는 **VectorStore** 인터페이스를 구현한다.
- 그래서 Langchain에서는 벡터 데이터베이스를 **Vector Store** 라고 한다.
- https://python.langchain.com/docs/integrations/vectorstores/

### Vector Store 연결
- Vector DB와 연결하는 메소드
- `VectorStore.from_documents()`
  - Document들을 insert 하면서 연결.
  - Database가 있으면 연결, 없으면 생성하면서 연결한다.
  - Parameter
    - documents: insert할 문서들을 list[Document]로 전달.
    - embedding model
    - vector db에 연결하기 위한 설정들을 넣어준다.
- `VectorStore()`
  - vector db와 연결만 한다.
  - Database가 있으면 연결, 없으면 생성하면서 연결한다.
  - Parameter
    - embedding model
    - vector db에 연결하기 위한 설정들을 넣어준다.
## InMemoryVectorStore
- langchain에서 제공하는 메모리 기반 벡터 데이터베이스이다.
- Data들을 Dictionary를 사용해 메모리에 저장하며, 검색 할 때 코사인 유사도(cosine similarity)를 계산하여 조회한다.

In [76]:
from langchain_core.documents import Document

# Upsert(SQL:insert) 할 Document 리스트 생성
d1 = Document(id=1, page_content="Apple, pear, Watermelon", metadata={"category":"fruit"})
d2 = Document(id=2, page_content="Python, Java, C++", metadata={"category":"IT"})
d3 = Document(id=3, page_content="Football, Baseball, BasketBall", metadata={"category":"Sport"})

In [77]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embedding_model = HuggingFaceEmbeddings(model="codefuse-ai/F2LLM-v2-1.7B")
# VectorStore를 생성 -> VectorDB와 연결
#                    -> 임베딩 모델을 넣어서 생성. (임베딩 벡터로 변환하는 것은 VectorStore가 담당.)
vectorstore = InMemoryVectorStore(embedding_model)

# Upsert - add_documents(list[Document])
# Document.page_content 를 Embedding Vector로 변환 -> index, Document를 value 저장
vectorstore.add_documents(documents=[d1, d2, d3])

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

['1', '2', '3']

In [88]:
# 검색(유사도기반)
query = "SQL"
query = "Rust"
query = "월드컵"
query = "아이폰"
result = vectorstore.similarity_search(
    query=query,    # 질의어
    k=2, # 몇개 문서를 조회할지 개수 (유사도 높은 순서로 k개를 반환.)
)

In [89]:
for doc in result:
    print(doc.page_content)

Apple, pear, Watermelon
Python, Java, C++


In [90]:
result = vectorstore.similarity_search_with_score(  # 유사도 점수를 포함해서 반환.
    query=query, k=2
)
for doc in result:
    print(doc)

(Document(id='1', metadata={'category': 'fruit'}, page_content='Apple, pear, Watermelon'), 0.2740858062853623)
(Document(id='2', metadata={'category': 'IT'}, page_content='Python, Java, C++'), 0.15742620574336022)


# 실습
- "data/olympic.txt" 문서를 vector store에 저장하고 질문과 유사한 청크를 조회
  
1. text loading
2. text split
3. embedding + vector store(InMemoryVectorStore)에 저장
4. query(질의)

In [1]:
from langchain_community.document_loaders import TextLoader # 문서로드
from dotenv import load_dotenv
# load_dotenv()

# Text Loading
path = "data/olympic.txt"
loader = TextLoader(path, encoding="utf-8")

C:\Users\Playdata\AppData\Local\Temp\ipykernel_18292\1812069810.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader # 문서로드


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # Chunking

# Text Split
## 청킹(스플릿) - 문서나누기
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50
)

# split_docs = text_splitter.split_documents(documents)
docs = loader.load_and_split(splitter)
print("로드된 문서개수:",len(docs))
# docs[0]

로드된 문서개수: 28


In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

##############################################################
# 정보 저장 - Indexing (Vector DB에 데이터들을 저장)
##############################################################

# 전처리 작업
# Vector DB에 저장
## Embedding모델, VectorStore 생성 => Document들을 저장
# embedding + vector store(InMemoryVectorStore)에 저장
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# Vectorstore 객체 생성 -> VectorDB와 연결.
vectorstore = InMemoryVectorStore(
    embedding=embeddings
)
# chunking한 문서들 저장 -> list[Document] : page_content의 내용을 embedding vector로 변환 후 저장.
vectorstore.add_documents(docs)

['c3489f5e-1761-436b-b9c3-ca8c66e9d542',
 '2d4f40d9-539e-42fc-9025-cca56594baa3',
 'ee24abad-ad86-4fc8-b36b-100855d2da32',
 '3b8f2289-6a66-40b6-b95f-f08afd89424b',
 '436c01c9-e71d-4b4a-bb8e-10892fb9113d',
 'f38be7a4-a701-42d9-932b-ce607a85daa5',
 '16117efa-5fa4-4adb-a678-0a1bc2eb0971',
 '1cdf72f9-f96d-408d-8389-ce94ad9765e4',
 'f34e7588-cffa-4a15-b20d-cda36185ec9e',
 '207a490f-1cf1-4aeb-ab21-5218a09e6c8f',
 '88dff2c0-0516-4e90-b4c0-2ac4fca7317c',
 '3c59b934-773b-46d0-8300-09ef39d24bea',
 '15f8fcc2-3c24-45d2-a379-6726cc729012',
 'a7611abc-9083-418f-b1bb-733cd6cf271b',
 '104048c0-b722-4033-98d5-5a2e2a7ffac5',
 'dc5aba82-dba2-4662-bf39-8eafbbe13b3f',
 'd0818715-3657-4ace-aa2e-8e4cdb1b1ff4',
 'b4bb3f8a-a7c2-4bf9-b754-8c683b7197de',
 '1496d593-7b49-4fa0-9dea-22625190b4b9',
 '5623e711-c0d0-41b9-9195-23d53a9c3ccc',
 'fe8366ec-7f3c-4d64-ac08-81b39d7cdecf',
 '1bb675fb-a7f3-4a80-a815-2afc45b591ba',
 '63817ac0-fcc8-45ff-923e-dadbeee23ef7',
 '7352650c-c559-4dd6-92f4-038bc0b60779',
 'ac8ffaec-7b97-

In [4]:
# doc들을 저장하면서 연결
## 위에 내용들을 합쳐서 간단하게
# vectorstore = InMemoryVectorStore.from_documents(
#     embedding=embeddings
#     documents=docs
# )

In [7]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

##############################################################################################
# 검색, 생성 
# - Vector DB에서 질문과 관련된 문서를 조회 -> 프롬프트=질문+찾은 문서들 -> LLM -> 응답
##############################################################################################

#query(질의)

query = "IOC는 어떤 기관인가요?"
# query = "올림픽이 취소된 경우가 있나요? 있다면 몇년도에 열린 몇회 올림픽인지, 그리고 취소 이유가 무엇인지 알려주세요"
# result_docs = vectorstore.similarity_search(
result_docs = vectorstore.similarity_search_with_score(     # (검색 결과, 유사도 점수)
    query,  # 질문 (질문을 embedding vector로 변환 후 검색까지 처리)
    k=5,    # 유사도 높은 순서로 k개반환, 반환할 문서개수.
)

In [8]:
result_context = []
for doc, score in result_docs:
    if score >= 0.4:
        result_context.append(doc)

In [9]:
result_context

[Document(id='207a490f-1cf1-4aeb-ab21-5218a09e6c8f', metadata={'source': 'data/olympic.txt'}, page_content='국제 올림픽 위원회\n올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다. 국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있다. 올림픽 활동은 크게 세 가지로 구성된다.\n- 국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구이다. 예를 들어서 국제 축구 연맹(FIFA)는 축구를 주관하며, 국제 배구 연맹(FIVB)은 배구를 주관하는 기구이다. 올림픽에는 현재 35개의 국제경기연맹이 있고 각 종목을 대표한다. (이 중에는 올림픽 종목은 아니지만 IOC의 승인을 받은 연맹도 있다.)\n- 국가 올림픽 위원회(NOC)는 각국의 올림픽 활동을 감독하는 기구이다. 예를 들어서 대한 올림픽 위원회(KOC)는 대한민국의 국가 올림픽 위원회이다. 현재 IOC에 소속된 국가 올림픽 위원회는 205개이다.\n- 올림픽 조직 위원회(OCOG)는 임시적인 조직으로 올림픽의 총체적인 것(개막식, 페막식 등)을 책임지기 위해 구성된 조직이다. 올림픽 조직 위원회는 올림픽이 끝나면 해산되며 최종보고서를 IOC에 제출한다.\n올림픽의 공식언어는 프랑스어와 영어와 개최국의 공용어이다. 모든 선언(예를 들어서 개막식 때 각국 소개를 할 때)들은 세 언어가 모두 나오거나 영어나 프랑스어 중에서 한 언어로만 말하기도 한다. 개최국의 공용어가 영어나 프랑스어가 아닐 때는 당연히 그 나라의 공용어도 함께 나온다.'),
 Document(id='2d4f40d9-539e-42fc-9025-cca56594baa3', metadata={'source': 'data/olympic.txt

In [10]:
# 프롬프트 구성 - 질문 - 검색한 문서들
prompt = ChatPromptTemplate.from_template(
    template="""<instruction>
당신은 친절한 QA Assistant 입니다.
질문에 대해 주어진 context를 기반으로 답을 해주세요.
context에 질문과 관련된 내용이 없을 경우 "주어진 정보로는 답을 알 수 없습니다." 라고 답을 하세요.
context에 없는 내용으로 답을 만들지 마세요.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
<output-format>
- 답변을 context의 어느부분을 참조하였는지 각주를 달아주세요.

예)
# 답변
최종 답변내용

# 참조내용
context에서 참조한 내용
</output-format>"""
)
model = ChatOpenAI(model="gpt-5.4-mini")
chain = prompt | model | StrOutputParser()

In [11]:
# Vector Store에서 조회한 결과(list[Document]) Document 에서 필요한 정보(page_context, metadata의 답변에 필요한 정보)만 추출
context = '\n\n'.join(doc.page_content for doc in result_context)

result = chain.invoke({"query":query, "context":context})

In [12]:
print(result)

# 답변
IOC는 **국제올림픽위원회**로, **모든 올림픽 활동을 통솔하는 단체**입니다. 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있습니다.[1]

# 참조내용
[1] “국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있다.”


In [15]:
result_context[0]

Document(id='51ade90d-a35c-4c8c-a623-5c42039a08ba', metadata={'source': 'data/olympic.txt'}, page_content='국제 올림픽 위원회\n올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다. 국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있다. 올림픽 활동은 크게 세 가지로 구성된다.\n- 국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구이다. 예를 들어서 국제 축구 연맹(FIFA)는 축구를 주관하며, 국제 배구 연맹(FIVB)은 배구를 주관하는 기구이다. 올림픽에는 현재 35개의 국제경기연맹이 있고 각 종목을 대표한다. (이 중에는 올림픽 종목은 아니지만 IOC의 승인을 받은 연맹도 있다.)\n- 국가 올림픽 위원회(NOC)는 각국의 올림픽 활동을 감독하는 기구이다. 예를 들어서 대한 올림픽 위원회(KOC)는 대한민국의 국가 올림픽 위원회이다. 현재 IOC에 소속된 국가 올림픽 위원회는 205개이다.\n- 올림픽 조직 위원회(OCOG)는 임시적인 조직으로 올림픽의 총체적인 것(개막식, 페막식 등)을 책임지기 위해 구성된 조직이다. 올림픽 조직 위원회는 올림픽이 끝나면 해산되며 최종보고서를 IOC에 제출한다.\n올림픽의 공식언어는 프랑스어와 영어와 개최국의 공용어이다. 모든 선언(예를 들어서 개막식 때 각국 소개를 할 때)들은 세 언어가 모두 나오거나 영어나 프랑스어 중에서 한 언어로만 말하기도 한다. 개최국의 공용어가 영어나 프랑스어가 아닐 때는 당연히 그 나라의 공용어도 함께 나온다.')

## MMR(최대 한계 관련성-Maximal Marginal Relevance) 알고리즘 적용
최대 한계 관련성(Maximal Marginal Relevance, MMR) 알고리즘은 정보 검색 및 요약에서 검색 결과의 **관련성**과 **다양성**을 동시에 고려하여 최적의 결과를 제공하는 방법이다. 
이 알고리즘은 사용자 쿼리와의 관련성을 최대화하면서도 중복 정보를 최소화하여 다양한 정보를 제공하는 것을 목표로 한다.

1. **관련성과 다양성의 균형 조절**: MMR은 사용자 쿼리와 문서 간의 유사성 점수와 이미 선택된 문서들과의 다양성 점수를 조합하여 각 문서의 최종 점수를 계산한다. 이를 통해 관련성이 높으면서도 중복되지 않는 문서를 선택한다.

2. **수학적 정의**
   $$
   \text{MMR} = \lambda \cdot \text{Sim}(d, Q) - (1 - \lambda) \cdot \max_{d' \in D'} \text{Sim}(d, d')
   $$

   - $\text{Sim}(d, Q)$: 문서 $d$와 쿼리 $\text{Q}$ 사이의 유사성. (문서 유사성 계산)
   - $\max_{d' \in D'} \text{Sim}(d, d')$: 문서 $d$와 이미 선택된 문서 집합 $D'$ 중 가장 유사한 문서와의 유사성. (문서 다양성 계산)
   - $\lambda$: 유사성과 다양성의 중요도를 조절하는 매개변수(parameter)
3. **적용 분야**: MMR은 정보 검색, 추천 시스템, 문서 요약 등에서 활용된다. 특히 LLM 검색에서 성능 향상이 입증되었다.

### `vectorStore.max_marginal_relevance_search()` 메소드
  - MMR 알고리즘을 적용한 검색을 수행한다.
  - **파라미터**
    - **query**: 사용자로부터 입력받은 검색 쿼리
    - **k**: 최종적으로 선택할 문서의 수
    - **fetch\_k**: MMR 알고리즘 적용 시 고려할 상위 문서의 수
    - **lambda_mult**: 쿼리와의 유사성과 선택된 문서 간의 다양성 사이의 균형을 조절하는 매개변수. $\lambda = 1$이면 유사성만 고려하고, $\lambda = 0$이면 다양성만을 최대화한다.
    - **filter**: 검색 결과를 필터링할 조건을 지정한다.


In [29]:
query = "IOC는 어떤 기구인가요?"
reuslt_docs_mmr = vectorstore.max_marginal_relevance_search(
    query=query, k=5,
    fetch_k=15,     # 다양성 계산을 위해서 상위 몇개 문서를 조회할지.
    lambda_mult=0.5 # 관련성과 다양성 사이의 비율
)

In [32]:
for doc in reuslt_docs_mmr:
    print(doc.page_content[:100])
    print("--------------------------------------------------------------")

국제 올림픽 위원회
올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다
--------------------------------------------------------------
우승자와 메달리스트
개인 혹은 팀으로 경기에 출전해서 1위, 2위, 3위를 한 선수는 메달을 받는다. 1912년까지는 우승자에게 순금으로 된 금메달을 주었으며 그 후에는 도금된 금
--------------------------------------------------------------
현재 이란 정부는 이스라엘과의 어떤 경기 경쟁이든 피하고 있다. 2008년 하계 올림픽 때 이란의 수영 선수는 이스라엘 수영 선수와 같이 경기한다는 이유로 경기를 포기했으며, 20
--------------------------------------------------------------
1990년대 후반, 여러 뜻있는 사람들이 도핑과의 전쟁을 선포하면서 1999년에 세계반도핑기구(WADA)를 설립한다. 2000년 하계 올림픽과 2002년 동계 올림픽 때는 약물 양
--------------------------------------------------------------
국제 올림픽 위원회(이하 IOC로 지칭)는 몇몇 위원들이 한 행위에 대해서 비판을 받고 있다. 그 예로 IOC 위원장이었던 에이버리 브런디지와 후안 안토니오 사마란치가 대표적인 사
--------------------------------------------------------------


In [33]:
for doc2 in vectorstore.similarity_search(query=query, k=5):
    print(doc2.page_content[:100])
    print("---------------------------------------------------------------------------------------------")

국제 올림픽 위원회
올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다
---------------------------------------------------------------------------------------------
또한 20세기에 올림픽 운동이 발전함에 따라, IOC는 변화하는 세계의 사회 환경에 적응해야 했다. 이러한 변화의 예로는 얼음과 눈을 이용한 경기 종목을 다루는 동계 올림픽, 장애
---------------------------------------------------------------------------------------------
2004년 10월과 11월에 IOC는 '올림픽 프로그램 위원회'(Olympic Programme Commission)를 설립했다. 여기서는 올림픽 종목과 올림픽 종목이 아닌 스포츠
---------------------------------------------------------------------------------------------
1990년대 후반, 여러 뜻있는 사람들이 도핑과의 전쟁을 선포하면서 1999년에 세계반도핑기구(WADA)를 설립한다. 2000년 하계 올림픽과 2002년 동계 올림픽 때는 약물 양
---------------------------------------------------------------------------------------------
올림픽 경기 종목
올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있다. 예를 들어서 하계 올림픽 부문인 레슬링은 자유형과 그레코로만형의 두 종목으로
---------------------------------------------------------------------------------------------
